# S22 · Let a solver find the best plan

Here is the one idea of this notebook, and it is a big one. For a whole family of
"find the best setting" problems, you do not have to invent a clever method
yourself. You just **describe the goal and the rules in plain terms**, and a solver
called **CVXPY** hands back the best plan. You write down what you want; CVXPY does
the searching.

We start with a plain business problem a Pune bakery owner would recognise: what to
bake this week to make the most profit, given only so much flour and so much oven
time. Then, in optional stretch cells at the end, we show that line-fitting, Lasso,
Ridge and a Nifty portfolio are all the very same kind of problem in disguise.

**New here? Read this once.**

- New to Python? You can still run every cell with the play button and read the
  plain-English note above each one. The main path below is short.
- New to the "bowl vs wobbly" idea? Open `primers/convex_sets_and_functions.md`.
- Already confident, or already know some optimisation? The main path is one simple
  bakery problem. Everything after it is marked **Stretch (optional)** — that is where
  line-fitting, Lasso, Ridge and a portfolio live.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. It installs CVXPY, the one library
Colab does not already have. On your **own machine** you installed everything with
`uv`, so the cell does nothing there.

In [ ]:
# CVXPY is the one library Colab does not already include. Install it only there.
import sys
if "google.colab" in sys.modules:
    !pip install -q cvxpy
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — import the tools we need

The usual NumPy and Matplotlib, plus **CVXPY**. Remember the big idea of CVXPY: you
*describe the problem* — the thing you want to make as good as possible, and any
rules it must obey — and CVXPY *finds the best answer* for you.

In [ ]:
import numpy as np                 # arrays and maths
import matplotlib.pyplot as plt    # simple charts
import cvxpy as cp                 # describe a goal + rules, get back the best plan

# Set a seed so anything random is the same every run.
np.random.seed(0)

print("Tools ready, including CVXPY.")

## Step 2 — the business problem

A Pune bakery is planning the week. It can bake two things: **trays of cakes** and
**trays of biscuits**. Two things run out: **flour** and **oven time**. This week the
bakery has **80 kg of flour** and **50 hours of oven time**, and no more.

From the recipes and the price list the owner knows:

- one **tray of cakes** uses **3 kg of flour** and **1 hour of oven**, and earns
  **Rs 400** profit,
- one **tray of biscuits** uses **2 kg of flour** and **3 hours of oven**, and earns
  **Rs 500** profit.

Biscuits earn more per tray, so should the bakery just make biscuits? Not
necessarily: biscuits are hungry for oven time, and cakes are cheap on the oven, so
the best plan is usually a mix. There are many possible weekly plans. Instead of
trying them by hand, we describe the goal and the rules and let the solver find the
best.

## Step 3 — describe the goal and the rules in CVXPY

Watch how closely the code reads like the paragraph above. We make one unknown for
the trays of each product, write the total profit we want to *maximise*, and list the
limits as plain inequalities. Then we hand the whole thing to CVXPY.

In [ ]:
# The unknowns: how many trays of each product to bake this week.
trays_of_cakes = cp.Variable()
trays_of_biscuits = cp.Variable()

# The goal: make the total weekly profit as large as possible.
total_profit = 400 * trays_of_cakes + 500 * trays_of_biscuits

# The rules the plan has to obey, one plain sentence each.
rules = [
    3 * trays_of_cakes + 2 * trays_of_biscuits <= 80,   # flour: only 80 kg available
    1 * trays_of_cakes + 3 * trays_of_biscuits <= 50,   # oven: only 50 hours available
    trays_of_cakes >= 0,                                 # cannot bake a negative amount
    trays_of_biscuits >= 0,
]

# Describe the problem (goal + rules) and let the solver find the best plan.
problem = cp.Problem(cp.Maximize(total_profit), rules)
problem.solve()

print("Solver status:", problem.status)

## Step 4 — read the answer, in trays and rupees

The solver has found the single best weekly plan. We read it straight back: how many
trays of each to bake, and the profit that plan earns, in rupees.

In [ ]:
best_cakes = round(float(trays_of_cakes.value))
best_biscuits = round(float(trays_of_biscuits.value))
best_profit = round(float(total_profit.value))

flour_used = 3 * best_cakes + 2 * best_biscuits
oven_used  = 1 * best_cakes + 3 * best_biscuits

print("Best plan for this week:")
print(f"   cakes    : {best_cakes} trays")
print(f"   biscuits : {best_biscuits} trays")
print()
print(f"Profit from this plan: Rs {best_profit:,}")
print()
print(f"Flour used: {flour_used} kg out of 80")
print(f"Oven used : {oven_used} hours out of 50")

## Step 5 — see the plan as a picture

A quick bar chart of the weekly plan. Notice the mix: the bakery does not pour
everything into the higher-earning biscuits, because the oven would run out first. A
blend of cakes and biscuits squeezes the most profit out of the same flour and oven.

In [ ]:
products = ["cakes", "biscuits"]
trays = [best_cakes, best_biscuits]

plt.figure(figsize=(7, 5))
bars = plt.bar(products, trays, color=["#2E75B6", "#1E8449"])
plt.ylabel("trays this week")
plt.title("The best weekly bake, found by the solver")
for bar, value in zip(bars, trays):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.3,
             f"{value} trays", ha="center")
plt.show()

## Step 6 — why we can trust this answer

Here is the payoff of the whole session. This bakery problem is **bowl-shaped**
(convex): the goal is a straight measure of profit and the rules are straight-line
limits, so the landscape of possible plans is a smooth bowl with a single best point.
That means the answer the solver returns is not just *a* good plan, it is the
*genuine best* one. Nothing better is hiding in some plan we did not try.

So the owner can say, with confidence, "this is the most profit our flour and oven
allow this week." That certainty is exactly what convexity buys you, and it is why
this family of problems is considered "solved". You describe the goal and the rules;
the solver finds the best.

## What you just did (this is the main point)

You took a real business decision, wrote down the **goal** (the most profit) and the
**rules** (the flour and the oven time you have), and let CVXPY find the best plan,
which you then read back in trays and rupees. You did not design a search method; you
just described what you wanted.

That is the transferable skill. The optional stretch cells below show that
line-fitting, Lasso, Ridge and a Nifty portfolio are all the **same bowl-shaped kind
of problem** — describe a goal and some rules, and a solver cracks it. Writing them
out in CVXPY by hand is a bonus, not something you need today.

### Stretch (optional) — the same idea as line-fitting (least squares)

Skip this if you are new; come back when you are curious. Fitting a straight line to
data is the *same* describe-the-goal move, only now the "plan" is the slope and
intercept of a line, and the goal is to make the total squared miss as small as
possible.

This is where the matrix notation from the maths shows up. We stack the data into a
table `X` (one row per flat, with a trailing 1 so the model can learn a starting
value), and write the total squared miss as the squared length `‖X·weights − y‖²`,
spelled `cp.sum_squares(X @ weights - y)` in CVXPY.

In [ ]:
# A small line-fitting dataset: flat size vs price, same idea as the regression session.
flat_size = np.linspace(1, 10, 40)
noise = np.random.normal(0, 3.5, size=40)
flat_price = 2.3 * flat_size + 4 + noise          # true rule: price = 2.3*size + 4, plus noise

# Stack the data as a table X: each row is [size, 1]. The 1 lets the line have a
# starting value b (the price when size is zero).
X = np.column_stack([flat_size, np.ones(40)])     # shape (40, 2)
y = flat_price

print("X shape:", X.shape)
print("first 3 rows of X:")
print(X[:3])

In [ ]:
# The plan is 'weights' = [slope w, starting value b]. The goal is the smallest
# total squared miss, ||X*weights - y||^2. There are no extra rules.
weights = cp.Variable(2)
squared_miss = cp.sum_squares(X @ weights - y)    # ||X*weights - y||^2

problem = cp.Problem(cp.Minimize(squared_miss))
problem.solve()

# Cross-check: the line-fitting session solved this with the normal equation
# (X^T X) weights = X^T y. NumPy gives the same numbers.
numpy_weights = np.linalg.solve(X.T @ X, X.T @ y)

print("CVXPY weights [w, b]:", weights.value.round(3))
print("NumPy weights [w, b]:", numpy_weights.round(3))
print()
print("They match - CVXPY solved the same line-fitting problem.")

In [ ]:
# A quick picture of the fitted line running through the flats.
predicted_price = weights.value[0] * flat_size + weights.value[1]

plt.figure(figsize=(7, 5))
plt.scatter(flat_size, flat_price, color="#2E75B6", label="real flats")
plt.plot(flat_size, predicted_price, color="#C0392B", linewidth=3, label="CVXPY line")
plt.xlabel("flat size")
plt.ylabel("flat price")
plt.title("Line-fitting solved by CVXPY (stretch)")
plt.legend()
plt.show()

### Stretch (optional) — Lasso: the solver decides which inputs matter

Skip this if you are new. **Lasso** is line-fitting with one extra term in the goal:
add a penalty `lambda·‖weights‖₁` (the sum of the *sizes* of the weights), written
`cp.norm(weights, 1)`. Its special power is switching useless weights to *exactly
zero*, so it tells you which inputs actually carry signal.

We build data with 8 input columns where only the first 3 truly drive the answer.
Think of predicting a loan default from 8 form fields when only 3 of them carry real
information. Lasso should keep those 3 and zero the rest.

In [ ]:
number_of_rows = 100
number_of_features = 8

# Random input table, and a target driven only by the first three columns.
features = np.random.normal(0, 1, size=(number_of_rows, number_of_features))
true_weights = np.array([4.0, -2.0, 3.0, 0.0, 0.0, 0.0, 0.0, 0.0])
target = features @ true_weights + np.random.normal(0, 0.5, size=number_of_rows)

# Goal = squared miss + lambda * (sum of the sizes of the weights).
lasso_weights = cp.Variable(number_of_features)
lam = 2.0
squared_miss = cp.sum_squares(features @ lasso_weights - target)
penalty = lam * cp.norm(lasso_weights, 1)          # the Lasso penalty

problem = cp.Problem(cp.Minimize(squared_miss + penalty))
problem.solve()

print("feature   Lasso weight   true weight")
for i in range(number_of_features):
    print(f"   {i}        {round(lasso_weights.value[i], 2):7.2f}       {true_weights[i]:5.1f}")

number_near_zero = int(np.sum(np.abs(lasso_weights.value) < 0.1))
print()
print("Weights Lasso pushed to (almost) zero:", number_near_zero, "out of", number_of_features)

In [ ]:
# Cross-check against scikit-learn's own Lasso (it scales the penalty by the number
# of rows, so we divide lambda by 2*rows to line the two versions up).
from sklearn.linear_model import Lasso

sklearn_lasso = Lasso(alpha=lam / (2 * number_of_rows), fit_intercept=False)
sklearn_lasso.fit(features, target)

print("CVXPY   Lasso weights:", lasso_weights.value.round(2))
print("sklearn Lasso weights:", sklearn_lasso.coef_.round(2))
print()
print("Both keep the first three features and zero out the noise ones.")

### Stretch (optional) — Ridge is the same recipe, one line changed

Skip this if you are new. Ridge is Lasso's sibling. The only change is the penalty:
instead of the sum of the *sizes* of the weights (`cp.norm(weights, 1)`), Ridge uses
the sum of the *squares* (`cp.sum_squares(weights)`). That one swap is the whole
difference. Ridge shrinks every weight toward zero but rarely snaps any to exactly
zero, unlike Lasso.

In [ ]:
ridge_weights = cp.Variable(number_of_features)
ridge_lambda = 2.0
squared_miss = cp.sum_squares(features @ ridge_weights - target)
ridge_penalty = ridge_lambda * cp.sum_squares(ridge_weights)   # the one-line change

ridge_problem = cp.Problem(cp.Minimize(squared_miss + ridge_penalty))
ridge_problem.solve()

print("Ridge weights:", ridge_weights.value.round(2))
print("Lasso weights:", lasso_weights.value.round(2))
print()
print("Ridge shrinks the noise weights but does not snap them to exactly zero;")
print("Lasso does. Same bowl-shaped recipe, different penalty, different behaviour.")

### Stretch (optional) — a Nifty portfolio is convex too

Skip this if you are new. This is the portfolio problem from the S16 session: given a
set of stocks, what fraction of your money goes into each to keep the risk (the
wobble of the combined return) as low as possible? That is the **Markowitz
portfolio**, and it is another bowl.

We invent seeded daily returns for 4 Nifty-style stocks, measure how they move
together (the covariance matrix), and ask CVXPY for the split `w` that minimises the
portfolio's variance `wᵀΣw` while spending all the money (`sum(w) = 1`) and never
going short (`w ≥ 0`). Nothing is downloaded; everything is synthetic and offline.

In [ ]:
number_of_days = 250
number_of_stocks = 4
daily_returns = np.random.normal(0.0006, 0.012, size=(number_of_days, number_of_stocks))

# Covariance matrix: how the 4 stocks wobble together (columns are stocks).
covariance = np.cov(daily_returns, rowvar=False)

# The plan: the fraction of money in each stock. Goal: smallest portfolio variance.
w = cp.Variable(number_of_stocks)
portfolio_risk = cp.quad_form(w, covariance)       # w^T Sigma w
rules = [cp.sum(w) == 1, w >= 0]                   # spend everything, no short selling

portfolio_problem = cp.Problem(cp.Minimize(portfolio_risk), rules)
portfolio_problem.solve()

print("Solver status:", portfolio_problem.status)
print("Lowest-risk split across the 4 stocks:")
for i in range(number_of_stocks):
    print(f"   stock {i}: {round(100 * w.value[i], 1)} % of the money")
print()
print("Fractions add up to:", round(float(np.sum(w.value)), 3), "(all the money invested).")

## The theme of this notebook

One move, over and over: **describe the goal and the rules, let the solver find the
best.** A bakery's week, a fitted line, Lasso, Ridge, a portfolio — all the same
bowl-shaped kind of problem, so one tool solves them all and you can trust the answer
is the genuine best. Next we meet a problem with proper fences, the support vector
machine, in `03_svm_the_widest_street.ipynb`.